# Assignment 3: Fine-tuning language models

In this assignment, you will perform supervised fine-tuning (SFT) of a small open LLM on an instruction tuning dataset. You will convert this dataset into instruction-response pairs, fine-tune a causal language model using LoRA (Low-Rank Adaptation), and evaluate it through prompted inference and comparison with other methods.

## Preliminaries

First, let's install the required libraries. If you are running in your own environment, make sure the following are installed:

- [Torch](https://docs.pytorch.org/docs/stable/index.html)
- [Transformers](https://huggingface.co/docs/transformers/index)
- [Datasets](https://huggingface.co/docs/datasets/index)
- [Evaluate](https://huggingface.co/docs/evaluate/en/index)
- [NLTK](https://www.nltk.org/api/nltk.html)
- [rouge_score](https://pypi.org/project/rouge-score/)

In a Colab notebook, most of them are already installed, except Evaluate and rouge_score.


In [1]:
%pip install evaluate rouge_score

Note: you may need to restart the kernel to use updated packages.


We also set some configuration parameters.

Most importantly, you should select a language model to work with in this assignment and enter its HuggingFace identifier in the parameter `MODEL_NAME` below. In principle you can use any model that you want, but we recommend that you select a model that has not already been trained to follow instructions, so it should be a "pure" language model trained on raw text (similar to Assignments 1 and 2).

The selected model should be small enough to fit in your computational environment. We have verified that the 135-million parameter [`SmolLM2` model](https://huggingface.co/HuggingFaceTB/SmolLM2-135M), developed by HuggingFace, can be used to solve this assignment in a Colab notebook (free tier, T4 GPU). If you run on a cluster, you can select a larger model (and probably see more interesting results).

We also define training and test set sizes here. Again, the values below have been set so that the assignment can be solved in Colab, and you can increase these sizes to improve the quality of the fine-tuned models.


In [2]:
import json
import math
import os
import random
import time

import torch

if torch.backends.mps.is_available():
    os.environ['ACCELERATE_MIXED_PRECISION'] = 'no'

SEED = 101
random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    DEVICE = 'cuda'
elif torch.backends.mps.is_available():
    DEVICE = 'mps'
else:
    DEVICE = 'cpu'

MAX_TRAIN_SAMPLES = 2000 if DEVICE == 'mps' else 5000
MAX_TEST_SAMPLES = 200 if DEVICE == 'mps' else 400
MAX_LENGTH = 128 if DEVICE == 'mps' else 384
MAX_NEW_TOKENS = 128

MODEL_NAME = 'HuggingFaceTB/SmolLM2-135M'
model_name_or_path = MODEL_NAME

USE_BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
USE_FP16 = torch.cuda.is_available() and not USE_BF16
MODEL_DTYPE = torch.float32 if DEVICE == 'mps' else torch.bfloat16 if USE_BF16 else torch.float32
TRAIN_BF16 = False if DEVICE == 'mps' else USE_BF16
TRAIN_FP16 = False if DEVICE == 'mps' else USE_FP16

print({'device': DEVICE, 'dtype': str(MODEL_DTYPE), 'model': MODEL_NAME})


{'device': 'mps', 'dtype': 'torch.float32', 'model': 'HuggingFaceTB/SmolLM2-135M'}


# Part 1: Preprocessing

### ⚙&nbsp; Task 1.1: Loading and inspecting the dataset

The dataset [SmolTalk](https://huggingface.co/datasets/HuggingFaceTB/smoltalk) is a collection of instruction-response pairs designed for SFT of large language models for instruction following. This dataset consists of examples of user inputs with system responses.

You can load using the datasets from the HuggingFace repository as follows.


In [3]:
from datasets import load_dataset
from datasets import DatasetDict

smoltalk = load_dataset("HuggingFaceTB/smoltalk", 'all')

/Users/telio/miniconda3/envs/phenoVLM-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In order to make this assignment possible to solve in a restricted environment, we simplify the dataset a bit:
- We remove multi-turn chat dialogues from the dataset;
- We remove instances where the query or the answer is greater than a set maximum length;
- We keep a subset of the data for training and testing (by default 5000 and 400, respectively).


In [4]:
smoltalk_simplified = smoltalk.filter(
    lambda row: len(row['messages']) <= 3 and all(len(m['content']) <= 256 for m in row['messages'])
)
smoltalk_simplified = smoltalk_simplified.shuffle(seed=SEED)
smoltalk_simplified = DatasetDict({
    'train': smoltalk_simplified['train'].select(range(min(MAX_TRAIN_SAMPLES, len(smoltalk_simplified['train'])))),
    'test': smoltalk_simplified['test'].select(range(min(MAX_TEST_SAMPLES, len(smoltalk_simplified['test'])))),
})


In [5]:
smoltalk_simplified

DatasetDict({
    train: Dataset({
        features: ['messages', 'source'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['messages', 'source'],
        num_rows: 200
    })
})

Print some examples from the dataset so that you understand the format.

Key points you need to note here: each example from the training or test set consists of a sequence of messages. The number of messages in each example will be 2 or 3, because we removed multi-turn chat dialogues in the previous step. Each message is associated with a `role` label:
- `user`: an example of something the user might write.
- `assistant`: an example of an output an LLM could be expected to produce, given the input.
- `system`: a *system prompt* that gives guidelines for the general behavior of the LLM's behavior.

All examples in the dataset include a user input and an assistant output, but the system prompt is not available in all of the examples.


In [6]:
smoltalk_simplified['train'][0]

{'messages': [{'content': 'Choose an appropriate phrase to integrate the sentences for better coherence:\nThe speaker highlighted the importance of climate change mitigation and urged the audience to take action. The talk was attended by government officials and experts in the field.',
   'role': 'user'},
  {'content': 'During his talk, the speaker highlighted the importance of climate change mitigation and urged the audience, including government officials and experts in the field, to take action.',
   'role': 'assistant'}],
 'source': 'explore-instruct-rewriting'}

### 🎓&nbsp; Task 1.2: Formatting the data for instruction tuning

**Why/how.** The selected base model, `HuggingFaceTB/SmolLM2-135M`, is a pure language model and does not define a tokenizer chat template. We therefore use one simple custom instruction template and reuse it consistently for training and generation. The important part is not the exact delimiter text, but that the model always sees the same boundary between user input and assistant response.

Define a function `format_input_output` that converts an example from the dataset into an input/output pair that we can use to fine-tune the LLM.

You are free to design the format. The following document gives some examples that have been used by different instruction-following LLMs including Llama and Mistral: https://huggingface.co/learn/llm-course/chapter11/2#common-template-formats

The later stages of our preprocessing pipeline expect that this function returns an object containing two parts: the `prompt` (what goes into the LLM before generating anything) and the `response` (what the LLM is expected to generate).


In [7]:
def make_prompt(user_text, system_text=None):
    """Custom instruction template for the non-chat SmolLM2 base model."""
    parts = []
    if system_text:
        parts.append(f'### System:\n{system_text.strip()}\n')
    parts.append(f'### User:\n{user_text.strip()}\n')
    parts.append('### Assistant:\n')
    return '\n'.join(parts)


def split_messages(messages):
    system = next((m['content'] for m in messages if m['role'] == 'system'), None)
    user = next(m['content'] for m in messages if m['role'] == 'user')
    assistant = next(m['content'] for m in messages if m['role'] == 'assistant')
    return system, user, assistant


def format_input_output(example):
    system, user, assistant = split_messages(example['messages'])
    return {
        'prompt': make_prompt(user, system),
        'response': assistant.strip(),
    }


Apply the function you implemented to the dataset as a whole.


In [8]:
ds_sft = smoltalk_simplified.map(format_input_output)

Then verify that the dataset now contains the new fields you created.


In [9]:
ds_sft['train'][0]

{'messages': [{'content': 'Choose an appropriate phrase to integrate the sentences for better coherence:\nThe speaker highlighted the importance of climate change mitigation and urged the audience to take action. The talk was attended by government officials and experts in the field.',
   'role': 'user'},
  {'content': 'During his talk, the speaker highlighted the importance of climate change mitigation and urged the audience, including government officials and experts in the field, to take action.',
   'role': 'assistant'}],
 'source': 'explore-instruct-rewriting',
 'prompt': '### User:\nChoose an appropriate phrase to integrate the sentences for better coherence:\nThe speaker highlighted the importance of climate change mitigation and urged the audience to take action. The talk was attended by government officials and experts in the field.\n\n### Assistant:\n',
 'response': 'During his talk, the speaker highlighted the importance of climate change mitigation and urged the audience, i

### ⚙&nbsp; Task 1.3: Tokenizing the dataset

We will now prepare the format required by the HuggingFace Trainer.

We first load the tokenizer for our selected model:


In [10]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

HAS_CHAT_TEMPLATE = bool(getattr(tokenizer, 'chat_template', None))
print({
    'pad_token': tokenizer.pad_token,
    'eos_token': tokenizer.eos_token,
    'has_chat_template': HAS_CHAT_TEMPLATE,
})
if HAS_CHAT_TEMPLATE:
    print('This model has a built-in chat template. For this assignment, prefer a non-instruction-tuned base model unless you intentionally want a chat-model baseline.')


{'pad_token': '<|endoftext|>', 'eos_token': '<|endoftext|>', 'has_chat_template': False}


Write a function `tokenize_helper` that takes an example (using the prompt/response format from the previous step) and produces the following three results:

- `input_ids`: the integer token ids of the concatenated prompt and response;
- `labels`: a list of the same length as `input_ids`, where the response token ids are the same, but where the prompt token ids have all been replaced by the loss masking identifier -100.
- `attention_mask`: the attention mask. This should just be a list of the same length as the other two lists, with all items set to 1.

The reason why `input_ids` and `labels` are different is that
we do not want to compute the training loss for tokens that appear in the user's input. We want to train the model to generate output *conditionally*: based on a prompt. But why the magic number -100? This is the number used by default in PyTorch's [`CrossEntropyLoss`](https://docs.pytorch.org/docs/stable/generated/torch.nn.CrossEntropyLoss.html) to indicate an item that should be excluded in loss computations. (This issue was also mentioned in [Assignment 1](https://liu-nlp.ai/dl4nlp/units/a1_1.html#task-4.1-implementing-the-trainer).)


In [11]:
def tokenize_helper(example):
    prompt = example['prompt']
    response = example['response']

    # We encode prompt and response separately so label masking lands exactly on
    # the assistant side of the training example.
    prompt_ids = tokenizer(prompt, add_special_tokens=False)['input_ids']
    response_text = response + (tokenizer.eos_token or '')
    response_ids = tokenizer(response_text, add_special_tokens=False)['input_ids']

    prefix_ids = []
    if tokenizer.bos_token_id is not None:
        prefix_ids.append(tokenizer.bos_token_id)

    max_prompt_len = max(0, MAX_LENGTH - len(prefix_ids) - 1)
    prompt_ids = prompt_ids[:max_prompt_len]
    response_budget = MAX_LENGTH - len(prefix_ids) - len(prompt_ids)
    response_ids = response_ids[:response_budget]

    input_ids = prefix_ids + prompt_ids + response_ids
    labels = [-100] * (len(prefix_ids) + len(prompt_ids)) + response_ids
    attention_mask = [1] * len(input_ids)

    return {
        'input_ids': input_ids,
        'attention_mask': attention_mask,
        'labels': labels,
    }


As above, apply the function you implemented to the dataset using `map`. This will add the three new fields to the dataset.


In [12]:
tokenized_ds_sft = ds_sft.map(
    tokenize_helper,
    remove_columns=ds_sft['train'].column_names,
)
print(tokenized_ds_sft)
print({k: len(tokenized_ds_sft['train'][0][k]) for k in ['input_ids', 'attention_mask', 'labels']})


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 2000
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels'],
        num_rows: 200
    })
})
{'input_ids': 86, 'attention_mask': 86, 'labels': 86}


## Part 2: Evaluation of the baseline model

As a first step, we will see how well the *baseline* model performs: that is, a model that has not been trained to follow instructions.


### ⚙&nbsp; Task 2.1: Preparing for evaluation

In this section, we set up a few utilities we will need to complete our training and evaluation infrastructure. These utilities will be given and you don't need to modify anything.

The first piece we need is a *collator*: that is, a tool that takes a number of instances and creates PyTorch tensors for a training batch. To make the batch fit into rectangular tensors, padding tokens will be added.


In [13]:
def data_collator(batch):
    """
    Create a custom collate function for causal language modeling.

    Args:
        batch: List of examples, each with 'input_ids', 'attention_mask', 'labels'
        tokenizer: Tokenizer with pad_token_id
    """

    input_ids_list = [torch.tensor(example["input_ids"], dtype=torch.long) for example in batch]
    attention_masks_list = [torch.tensor(example["attention_mask"], dtype=torch.long) for example in batch]
    labels_list = [torch.tensor(example['labels'], dtype=torch.long) for example in batch]

    # Find max length in this batch
    max_len = max(x.size(0) for x in input_ids_list)

    # Helper pad function
    def pad_to_max(x_list, pad_value):
        padded = []
        for x in x_list:
            pad_len = max_len - x.size(0)
            if pad_len > 0:
                pad_tensor = torch.full((pad_len,), pad_value, dtype=x.dtype)
                x = torch.cat([x, pad_tensor], dim=0)
            padded.append(x)
        return torch.stack(padded, dim=0)

    # Use tokenizer.pad_token_id for inputs, 0 for attention_mask, -100 for labels
    pad_id = tokenizer.pad_token_id

    batch_input_ids = pad_to_max(input_ids_list, pad_value=pad_id)
    batch_attention_mask = pad_to_max(attention_masks_list, pad_value=0)
    batch_labels = pad_to_max(labels_list, pad_value=-100)

    batch = {
            "input_ids": batch_input_ids,
            "attention_mask": batch_attention_mask,
            "labels": batch_labels,
        }
    return batch

The second utility we need is an evaluator. We will use the **ROUGE-L** metric, which computes the longest common subsequence between the model's output and the gold-standard answer. You can read about ROUGE-L here: https://en.wikipedia.org/wiki/ROUGE_(metric)

When using the ROUGE-L metric in a Trainer, we need to wrap it in an object defined as follows:


In [14]:
import evaluate


class RougeMetricComputer:
    """Accumulate decoded response-token predictions and compute ROUGE-L."""

    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
        self.rouge = evaluate.load('rouge')
        self.all_predictions = []
        self.all_references = []

    def __call__(self, eval_pred, compute_result=False):
        logits, labels = eval_pred
        pred_ids = logits.argmax(axis=-1)

        for pred, label in zip(pred_ids, labels):
            mask = label != -100
            if mask.sum() == 0:
                continue

            ref_ids = label[mask]
            pred_ids_filtered = pred[mask]
            ref_text = self.tokenizer.decode(ref_ids, skip_special_tokens=True)
            pred_text = self.tokenizer.decode(pred_ids_filtered, skip_special_tokens=True)
            self.all_references.append(ref_text.strip())
            self.all_predictions.append(pred_text.strip())

        if not compute_result:
            return {}

        if not self.all_references:
            return {'rougeL': 0.0}

        scores = self.rouge.compute(
            predictions=self.all_predictions,
            references=self.all_references,
        )
        self.all_predictions = []
        self.all_references = []
        return {'rougeL': scores['rougeL']}


compute_metrics = RougeMetricComputer(tokenizer)


Finally, we make a function that sets up a [`Trainer`](https://huggingface.co/docs/transformers/main_classes/trainer).


In [15]:
from transformers import Trainer
from transformers.trainer_callback import ProgressCallback

def make_trainer(model, training_args):
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_ds_sft["train"],
        eval_dataset=tokenized_ds_sft["test"],
        compute_metrics=compute_metrics,
        data_collator=data_collator,
    )
    trainer.callback_handler.callbacks = [
        cb for cb in trainer.callback_handler.callbacks
        if type(cb).__name__ != "NotebookProgressCallback"
    ]
    trainer.add_callback(ProgressCallback)
    return trainer


### 🎓&nbsp; Task 2.2: Evaluating the pre-trained model

**Why/how.** The pre-trained model has learned general next-token prediction, but it has not been adapted to this prompt format. This baseline tells us how much instruction tuning changes loss and ROUGE-L before we compare full SFT and LoRA.

Now, we have all the pieces to evaluate our baseline model that has not been instruction-tuned.

Why do you think the ROUGE-L score is as high as it is, even without any training for instruction-following?

A non-instruction-tuned model can still get nonzero ROUGE-L because it has learned general language patterns and common answer phrasing during pretraining. ROUGE-L measures token overlap, not whether the model truly follows the instruction.


In [16]:
from transformers import TrainingArguments
from transformers import AutoModelForCausalLM

print('\n' + '=' * 80)
print('EVALUATING PRETRAINED MODEL')
print('=' * 80)

pretrained_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=MODEL_DTYPE,
).to(DEVICE)
pretrained_model.config.pad_token_id = tokenizer.pad_token_id

pretrained_eval_args = TrainingArguments(
    output_dir='results/assignments/a3_pretrained_eval',
    eval_strategy='no',
    per_device_eval_batch_size=1,
    bf16=TRAIN_BF16,
    fp16=TRAIN_FP16,
    report_to='none',
    optim='adafactor',
    batch_eval_metrics=True,
    eval_accumulation_steps=1,
)

pretrained_trainer = make_trainer(pretrained_model, pretrained_eval_args)

t0 = time.perf_counter()
pretrained_eval_metrics = pretrained_trainer.evaluate()
pretrained_eval_time = time.perf_counter() - t0

pretrained_eval_loss = float(pretrained_eval_metrics['eval_loss'])
pretrained_rougeL = pretrained_eval_metrics.get('eval_rougeL', None)

print('\nPRETRAINED EVAL METRICS:')
print(json.dumps(pretrained_eval_metrics, indent=2))
print('eval time:', round(pretrained_eval_time, 2), 'seconds')


`torch_dtype` is deprecated! Use `dtype` instead!



EVALUATING PRETRAINED MODEL


/Users/telio/miniconda3/envs/phenoVLM-env/lib/python3.12/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)
100%|██████████| 200/200 [00:24<00:00,  8.32it/s]


PRETRAINED EVAL METRICS:
{
  "eval_loss": 1.939670205116272,
  "eval_model_preparation_time": 0.0015,
  "eval_rougeL": 0.5792921161611353,
  "eval_runtime": 24.3249,
  "eval_samples_per_second": 8.222,
  "eval_steps_per_second": 8.222
}
eval time: 24.33 seconds


## Part 3: Supervised fine-tuning


### 🎓&nbsp; Task 3.1: Training the full model

**Why/how.** Full SFT updates every model parameter, so it is the direct but expensive way to adapt the base LM to the instruction-response format. Comparing against the baseline shows whether the model learned the desired assistant behavior.

Next, we train the pre-trained model using SFT over all the parameters, then calculate the metrics and outputs to evaluate how well it follows instructions.

How do the results differ from those in the previous step?

After full SFT, the expected pattern is lower evaluation loss and more prompt-aware generations. The actual ROUGE-L change can be modest because it is a surface-overlap metric and the model is small.


In [17]:
FULL_SFT_EPOCHS = 1
FULL_SFT_BATCH_SIZE = 1

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=MODEL_DTYPE,
).to(DEVICE)
base_model.config.pad_token_id = tokenizer.pad_token_id
base_model.gradient_checkpointing_enable()
base_model.config.use_cache = False

baseline_training_args = TrainingArguments(
    output_dir='results/assignments/a3_full_sft',
    eval_strategy='epoch',
    logging_steps=200,
    save_strategy='no',
    num_train_epochs=FULL_SFT_EPOCHS,
    per_device_train_batch_size=FULL_SFT_BATCH_SIZE,
    per_device_eval_batch_size=1,
    bf16=TRAIN_BF16,
    fp16=TRAIN_FP16,
    report_to='none',
    optim='adafactor',
    batch_eval_metrics=True,
    eval_accumulation_steps=1,
    gradient_checkpointing=True,
)

baseline_trainer = make_trainer(base_model, baseline_training_args)

t0 = time.perf_counter()
baseline_train_output = baseline_trainer.train()
baseline_train_time = time.perf_counter() - t0
baseline_eval_metrics = baseline_trainer.evaluate()

print('full SFT train time:', round(baseline_train_time, 2), 'seconds')
print(json.dumps(baseline_eval_metrics, indent=2))


 10%|█         | 200/2000 [01:46<14:07,  2.12it/s]

{'loss': 1.5372, 'grad_norm': 5.04132080078125, 'learning_rate': 4.5025000000000003e-05, 'epoch': 0.1}


 20%|██        | 400/2000 [03:21<13:00,  2.05it/s]

{'loss': 1.3112, 'grad_norm': 14.25717544555664, 'learning_rate': 4.0025000000000004e-05, 'epoch': 0.2}


 30%|███       | 600/2000 [04:51<10:11,  2.29it/s]

...

{'loss': 1.3504, 'grad_norm': 12.820672988891602, 'learning_rate': 2.5000000000000002e-08, 'epoch': 1.0}


                                                   
100%|██████████| 2000/2000 [14:57<00:00,  2.23it/s]


{'eval_loss': 1.2301030158996582, 'eval_rougeL': 0.6373278796917685, 'eval_runtime': 14.3851, 'eval_samples_per_second': 13.903, 'eval_steps_per_second': 13.903, 'epoch': 1.0}
{'train_runtime': 897.6825, 'train_samples_per_second': 2.228, 'train_steps_per_second': 2.228, 'train_loss': 1.3708865661621095, 'epoch': 1.0}


100%|██████████| 200/200 [00:12<00:00, 15.46it/s]

full SFT train time: 897.9 seconds
{
  "eval_loss": 1.2301030158996582,
  "eval_rougeL": 0.6373278796917685,
  "eval_runtime": 13.0085,
  "eval_samples_per_second": 15.375,
  "eval_steps_per_second": 15.375,
  "epoch": 1.0
}


### ⚙&nbsp; Task 3.3: Counting the number of trainable parameters

Define a function `num_trainable_parameters` that computes the number of floating-point numbers that a given model will update during training.

**Hints**:
- For a PyTorch module `m`, you can use `m.parameters()` to access its parameter tensors.
- However, you should only include parameter tensors where the flag `requires_grad` is True.


In [18]:
def num_trainable_parameters(model):
    """Count the number of trainable scalar parameters in a PyTorch module."""
    return sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)


print('full SFT trainable parameters:', num_trainable_parameters(base_model))


full SFT trainable parameters: 134515008


Apply this function to the SFT-trained model and check that the result makes sense.


## Part 4: Parameter-efficient fine-tuning

In the last section of this assignment, we will use LoRA to train the model in a more parameter-efficient manner. You may want to prepare by reading  by [the paper by Hu et al. (2021)](https://arxiv.org/pdf/2106.09685) and the teaching material provided for this course.

### ⚙&nbsp; Task 4.1: Utilities for modifying models

Define a function `extract_lora_targets` that extracts the relevant linear layers from all Transformer blocks in your selected LLM.
It is up to you to decide what layers to select; in the experiments described in the original LoRA paper, the query and value projection matrices were fine-tuned with LoRA, while all other layers were left unchanged.
Return a dictionary that maps the component name to the corresponding linear layer.

As we saw earlier (in Assignment 2 and elsewhere), a Transformer model consists of a hierarchy of nested submodules. Each of these can be addressed by a fully-qualified string name. You can use get_submodule() to retrieve a layer by a string name. This name depends on the model you have selected. For instance, in the `SmolLM2-135M` model, `'model.layers.0.self_attn.q_proj'`
 refers to the query projection in Transformer layer 0.

It is OK to hard-code this part, so that you just enumerate the layers you want to extract. Alternatively, use a utility such as `model.named_modules()` to iterate through the model's layers.


In [19]:
def extract_lora_targets(model):
    """Return attention projection layers suitable for LoRA replacement."""
    target_suffixes = ('q_proj', 'k_proj', 'v_proj', 'o_proj')
    return {
        name: module
        for name, module in model.named_modules()
        if isinstance(module, torch.nn.Linear) and name.endswith(target_suffixes)
    }


We also need a convenience function that puts layers back into a model. The following function does the trick. The `named_layers` argument uses the same format as returned by `extract_lora_targets`.


In [20]:
def replace_layers(model, named_layers):
    """
    Replace submodules in `model` by name.
    """
    for name, layer in named_layers.items():
        components = name.split(".")
        submodule = model
        for comp in components[:-1]:
            submodule = getattr(submodule, comp)
        setattr(submodule, components[-1], layer)
    return model

### 🎓&nbsp; Task 4.2: Implementing the LoRA layer

**Why/how.** LoRA freezes the original linear layer and learns a low-rank update `B(Ax)`. This reduces trainable parameters while still allowing the model to change important attention projections.

To implement the LoRA approach, we define a new type of layer that will be used as a drop-in replacement for a regular linear layer.

In [the paper by Hu et al. (2021)](https://arxiv.org/pdf/2106.09685), the structure is presented visually in Figure 1, and equation (3) shows the same idea.


In [21]:
import torch.nn as nn


class LoRALayer(nn.Module):
    def __init__(self, W, r, alpha):
        super().__init__()
        if r <= 0:
            raise ValueError('r must be positive')

        self.W = W
        for parameter in self.W.parameters():
            parameter.requires_grad = False

        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r
        self.lora_A = nn.Linear(W.in_features, r, bias=False)
        self.lora_B = nn.Linear(r, W.out_features, bias=False)

        nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B.weight)

    def forward(self, x):
        return self.W(x) + self.scaling * self.lora_B(self.lora_A(x))


Here, `W` is the linear layer we are fine-tuning, while `r` and `alpha` are hyperparameters described in section 4.1. of the paper. The `r` parameter controls the parameter efficiency: by setting it to a low value, we save memory but make a rougher approximation. The `alpha` parameter is a scaling factor.


### 🎓&nbsp; Task 4.3: Fine-tuning with LoRA

**Why/how.** The LoRA model keeps the base model frozen and updates only the low-rank adapter matrices. It should train fewer parameters than full SFT, and the comparison tells us how much adaptation quality we keep for the smaller update budget.

Set up a model where you replace the four linear layers in attention blocks (query, key, value, and output) with LoRA layers. Use the following steps:
- First use `extract_lora_targets` to get the relevant linear layers.
- Each of the linear layers in the returned dictionary should be wrapped inside a LoRA layer.
- Then use `replace_layers` to put them back into the model.

Train this model and compare the training speed, metrics, and outputs to the results from Part 3.

LoRA should update far fewer parameters than full SFT, usually reducing memory use and making training cheaper. With a small rank it may not fully match full SFT quality, but it should still improve over the raw pretrained baseline.


In [22]:
LORA_R = 8
LORA_ALPHA = 16
LORA_EPOCHS = 1
LORA_BATCH_SIZE = 1

lora_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=MODEL_DTYPE,
).to(DEVICE)
lora_model.config.pad_token_id = tokenizer.pad_token_id

for parameter in lora_model.parameters():
    parameter.requires_grad = False

target_layers = extract_lora_targets(lora_model)
print('LoRA target layers:', len(target_layers))
lora_layers = {
    name: LoRALayer(layer, r=LORA_R, alpha=LORA_ALPHA).to(DEVICE)
    for name, layer in target_layers.items()
}
lora_model = replace_layers(lora_model, lora_layers)
print('LoRA trainable parameters:', num_trainable_parameters(lora_model))

if hasattr(lora_model, 'enable_input_require_grads'):
    lora_model.enable_input_require_grads()

lora_model.gradient_checkpointing_enable()
lora_model.config.use_cache = False

lora_training_args = TrainingArguments(
    output_dir='results/assignments/a3_lora',
    eval_strategy='epoch',
    logging_steps=200,
    save_strategy='no',
    num_train_epochs=LORA_EPOCHS,
    per_device_train_batch_size=LORA_BATCH_SIZE,
    per_device_eval_batch_size=1,
    bf16=TRAIN_BF16,
    fp16=TRAIN_FP16,
    report_to='none',
    optim='adafactor',
    batch_eval_metrics=True,
    eval_accumulation_steps=1,
    gradient_checkpointing=True,
)

lora_trainer = make_trainer(lora_model, lora_training_args)

t0 = time.perf_counter()
lora_train_output = lora_trainer.train()
lora_train_time = time.perf_counter() - t0
lora_eval_metrics = lora_trainer.evaluate()

print('LoRA train time:', round(lora_train_time, 2), 'seconds')
print(json.dumps(lora_eval_metrics, indent=2))


LoRA target layers: 120
LoRA trainable parameters: 921600


 10%|█         | 200/2000 [12:46<11:46,  2.55it/s]   

{'loss': 1.7957, 'grad_norm': 0.5668878555297852, 'learning_rate': 4.5025000000000003e-05, 'epoch': 0.1}


 20%|██        | 400/2000 [1:18:21<11:17,  2.36it/s]     

{'loss': 1.4605, 'grad_norm': 2.505462646484375, 'learning_rate': 4.0025000000000004e-05, 'epoch': 0.2}


...

{'loss': 1.5371, 'grad_norm': 2.209627628326416, 'learning_rate': 2.5000000000000002e-08, 'epoch': 1.0}


                                                      
100%|██████████| 2000/2000 [11:51:28<00:00, 21.34s/it]


{'eval_loss': 1.4175024032592773, 'eval_rougeL': 0.6152913389242726, 'eval_runtime': 1967.1857, 'eval_samples_per_second': 0.102, 'eval_steps_per_second': 0.102, 'epoch': 1.0}
{'train_runtime': 42688.7054, 'train_samples_per_second': 0.047, 'train_steps_per_second': 0.047, 'train_loss': 1.531361572265625, 'epoch': 1.0}


100%|██████████| 200/200 [17:49<00:00,  5.35s/it]   

LoRA train time: 930.13 seconds
{
  "eval_loss": 1.4175024032592773,
  "eval_rougeL": 0.6152913389242726,
  "eval_runtime": 1069.9634,
  "eval_samples_per_second": 0.187,
  "eval_steps_per_second": 0.187,
  "epoch": 1.0
}


### 🎓&nbsp; Task 4.4: Qualitative inspection

**Why/how.** Aggregate metrics are useful, but instruction tuning is ultimately about behavior. Prompting the baseline, full-SFT, and LoRA models on the same inputs lets us inspect whether the models follow the requested format and answer the user query sensibly.

Run the three models interactively on some examples of your own choice (either taken from the training or test sets, or created by yourself). The convenience function below can be of use, but you need to complete it by using the prompt format you defined in Task 1.2.

Do your models seem to have learned the instruction-following behavior (at least to some extent)? Do they respond to user queries sensibly?

Good signs are direct answers, fewer prompt continuations, and responses that stay on the requested task. Full SFT should usually show this most clearly, with LoRA often close if the rank and training budget are sufficient.


In [23]:
@torch.no_grad()
def generate_response(model, user_text, system_text=None, max_new_tokens=MAX_NEW_TOKENS):
    model.eval()
    prompt = make_prompt(user_text, system_text)
    inputs = tokenizer(prompt, return_tensors='pt').to(DEVICE)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    new_ids = output_ids[0, inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_ids, skip_special_tokens=True).strip()


examples = [
    split_messages(ds_sft['test'][0]['messages'])[:2],
    (None, 'Explain why the sky looks blue in two sentences.'),
]

for system_text, user_text in examples:
    print('\nUSER:', user_text)
    for name, model in [
        ('pretrained', pretrained_model),
        ('full_sft', base_model),
        ('lora', lora_model),
    ]:
        print(f'{name}:', generate_response(model, user_text, system_text))



USER: Add a transitional expression to connect two ideas that belong to different sentences:
The first step in this process is to gather the necessary materials. Once you have them, you can proceed to the next step.
pretrained: You are an AI assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.

### User:
Add a transitional expression to connect two ideas that belong to different sentences:
The first step in this process is to gather the necessary materials. Once you have them, you can proceed to the next step.

### Assistant:
You are an AI assistant. You will be provided with a text and you need to rewrite it according to the user's instructions.

### User:
Add a transitional expression to connect two ideas that belong to different sentences:
The first
full_sft: To begin with, you need to gather the necessary materials. Once you have them, you can proceed to the next step.
lora: The first step in this process is to gather the nec